# SpiritPool Signal Explorer

Debug and analyze contributor signals flowing through the SpiritPool pipeline.

**A/B comparison:** Raw signals (sp_events.payload) vs Processed output (job_postings).

**Isolation:** This notebook is read-only. No production imports. Direct SQL only.

In [ ]:
import pandas as pd
import json
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='../../.env')
DB_URL = os.environ.get('DATABASE_URL', 'sqlite:///../../data/tracker.db')

engine = create_engine(DB_URL, echo=False)

def q(sql, **params):
    """Run a read-only query, return DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

def expand_payload(df, col='payload'):
    """Expand a JSONB/JSON column into separate DataFrame columns."""
    parsed = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else (x or {}))
    expanded = pd.json_normalize(parsed)
    return pd.concat([df.drop(columns=[col]), expanded], axis=1)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)

# Mask credentials in connection string for display
display_url = DB_URL.split('@')[-1] if '@' in DB_URL else DB_URL
print(f'Connected to: {display_url}')

---
## 1. Overview — Row Counts & Freshness

In [ ]:
overview = q("""
SELECT 'sp_events' AS table_name,
       COUNT(*)    AS rows,
       MAX(collected_at)::text AS latest
FROM sp_events
UNION ALL
SELECT 'quarantine',
       COUNT(*),
       MAX(quarantined_at)::text
FROM quarantine
UNION ALL
SELECT 'job_postings (spiritpool)',
       COUNT(*),
       MAX(scraped_at)::text
FROM job_postings
WHERE source LIKE 'spiritpool_%%'
""")
print('Pipeline overview:')
overview

In [ ]:
# Breakdown by source_type (extension_legacy vs extension)
q("""
SELECT source_type, COUNT(*) AS events, 
       MIN(collected_at)::text AS earliest,
       MAX(collected_at)::text AS latest
FROM sp_events
GROUP BY source_type
ORDER BY events DESC
""")

---
## 2. Raw Signals — sp_events (A side)

In [ ]:
# All sp_events, most recent first — expand payload JSONB into columns
raw = q("""
SELECT event_id, session_token, epoch_id, event_type, source_type,
       collected_at, pipeline_version, payload
FROM sp_events
ORDER BY collected_at DESC
LIMIT 50
""")

if len(raw) > 0:
    raw_expanded = expand_payload(raw)
    print(f'{len(raw)} signals (showing latest 50)')
    display(raw_expanded)
else:
    print('No signals received yet.')

In [ ]:
# Filter: LinkedIn signals only
linkedin_raw = q("""
SELECT event_id, session_token, epoch_id, collected_at, payload
FROM sp_events
WHERE payload->>'legacy_domain' = 'linkedin'
ORDER BY collected_at DESC
LIMIT 50
""")

if len(linkedin_raw) > 0:
    linkedin_expanded = expand_payload(linkedin_raw)
    print(f'{len(linkedin_raw)} LinkedIn signals')
    display(linkedin_expanded)
else:
    print('No LinkedIn signals in sp_events yet.')
    print('Tip: Check if signals arrived via a different domain tag:')
    display(q("""
        SELECT payload->>'legacy_domain' AS domain, COUNT(*) AS n
        FROM sp_events
        GROUP BY payload->>'legacy_domain'
        ORDER BY n DESC
    """))

---
## 3. Processed Output — job_postings (B side)

In [ ]:
processed = q("""
SELECT id, source, raw_employer_name, normalized_name, role_title,
       wage_min, wage_max, wage_period,
       raw_address, lat, lng, region,
       source_url, scraped_at, is_active,
       match_method, match_confidence
FROM job_postings
WHERE source LIKE 'spiritpool_%%'
ORDER BY scraped_at DESC
LIMIT 50
""")

if len(processed) > 0:
    print(f'{len(processed)} processed job postings from SpiritPool')
    display(processed)
else:
    print('No SpiritPool job postings ingested yet.')

In [ ]:
# LinkedIn-specific processed output
linkedin_processed = q("""
SELECT id, raw_employer_name, normalized_name, role_title,
       wage_min, wage_max, wage_period,
       raw_address, lat, lng,
       source_url, scraped_at
FROM job_postings
WHERE source = 'spiritpool_linkedin'
ORDER BY scraped_at DESC
LIMIT 50
""")

print(f'{len(linkedin_processed)} LinkedIn job postings')
linkedin_processed

---
## 4. A/B Comparison — Raw vs Processed

Join sp_events (raw) to job_postings (processed) to see how signals transform through the pipeline.

In [ ]:
ab = q("""
SELECT
    e.event_id,
    e.collected_at                          AS raw_collected_at,
    e.payload->>'company'                   AS raw_company,
    j.raw_employer_name                     AS proc_employer,
    j.normalized_name                       AS proc_normalized,
    e.payload->>'jobTitle'                  AS raw_title,
    j.role_title                            AS proc_title,
    e.payload->'salary'->>'min'             AS raw_salary_min,
    j.wage_min                              AS proc_wage_min,
    e.payload->'salary'->>'max'             AS raw_salary_max,
    j.wage_max                              AS proc_wage_max,
    e.payload->'salary'->>'period'          AS raw_wage_period,
    j.wage_period                           AS proc_wage_period,
    e.payload->>'location'                  AS raw_location,
    j.raw_address                           AS proc_address,
    j.lat                                   AS proc_lat,
    j.lng                                   AS proc_lng,
    j.match_method,
    j.match_confidence
FROM sp_events e
JOIN job_postings j
  ON j.source LIKE 'spiritpool_%%'
 AND j.raw_employer_name = e.payload->>'company'
 AND j.role_title        = e.payload->>'jobTitle'
 AND ABS(EXTRACT(EPOCH FROM j.scraped_at - e.collected_at)) < 300
ORDER BY e.collected_at DESC
LIMIT 100
""")

if len(ab) > 0:
    print(f'{len(ab)} matched raw→processed pairs')
    display(ab)
else:
    print('No matched pairs found.')
    print(f'  sp_events rows:   {q("SELECT COUNT(*) AS n FROM sp_events").iloc[0,0]}')
    print(f'  spiritpool jobs:  {q("SELECT COUNT(*) AS n FROM job_postings WHERE source LIKE \'spiritpool_%%\'").iloc[0,0]}')

In [ ]:
# Mismatch detection: signals in sp_events with no matching job_posting
orphans = q("""
SELECT e.event_id, e.collected_at,
       e.payload->>'company'   AS company,
       e.payload->>'jobTitle'  AS job_title,
       e.payload->>'legacy_domain' AS domain
FROM sp_events e
LEFT JOIN job_postings j
  ON j.source LIKE 'spiritpool_%%'
 AND j.raw_employer_name = e.payload->>'company'
 AND j.role_title        = e.payload->>'jobTitle'
 AND ABS(EXTRACT(EPOCH FROM j.scraped_at - e.collected_at)) < 300
WHERE j.id IS NULL
ORDER BY e.collected_at DESC
LIMIT 25
""")

if len(orphans) > 0:
    print(f'{len(orphans)} sp_events with NO matching job_posting (investigate):')
    display(orphans)
else:
    print('All sp_events have a matching job_posting.')

---
## 5. Quarantined Signals — PII Detection

In [ ]:
quarantined = q("""
SELECT quarantine_id, redaction_types, quarantined_at, rule_version,
       original_payload
FROM quarantine
ORDER BY quarantined_at DESC
LIMIT 25
""")

if len(quarantined) > 0:
    print(f'{len(quarantined)} quarantined signals')
    # Show summary without full payload
    summary = quarantined[['quarantine_id', 'redaction_types', 'quarantined_at', 'rule_version']].copy()
    display(summary)
    
    # PII type breakdown
    print('\nPII detection breakdown:')
    display(q("""
        SELECT redaction_types, COUNT(*) AS count
        FROM quarantine
        GROUP BY redaction_types
        ORDER BY count DESC
    """))
else:
    print('No quarantined signals.')

In [ ]:
# Inspect quarantined payloads — what fields triggered PII?
if len(quarantined) > 0:
    print('Quarantined payload inspection (first 5):\n')
    for _, row in quarantined.head(5).iterrows():
        payload = row['original_payload']
        if isinstance(payload, str):
            payload = json.loads(payload)
        print(f"ID: {row['quarantine_id'][:12]}...  Types: {row['redaction_types']}")
        print(json.dumps(payload, indent=2, default=str)[:500])
        print('---')

---
## 6. Field Stripping Verification

Confirm no forbidden fields leaked into stored payloads.

In [ ]:
forbidden_checks = {
    'tabUrl':        "SELECT COUNT(*) AS n FROM sp_events WHERE payload ? 'tabUrl'",
    'collectedAt':   "SELECT COUNT(*) AS n FROM sp_events WHERE payload ? 'collectedAt'",
    'consent_state': "SELECT COUNT(*) AS n FROM sp_events WHERE payload ? 'consent_state'",
}

print('Field stripping verification:')
all_pass = True
for field, sql in forbidden_checks.items():
    count = q(sql).iloc[0, 0]
    status = 'PASS' if count == 0 else f'VIOLATION ({count} rows)'
    if count > 0:
        all_pass = False
    print(f'  {field:<20} {status}')

# Also check quarantine table
for field in ['tabUrl', 'collectedAt', 'consent_state']:
    count = q(f"SELECT COUNT(*) AS n FROM quarantine WHERE original_payload ? '{field}'").iloc[0, 0]
    status = 'PASS' if count == 0 else f'VIOLATION ({count} rows)'
    if count > 0:
        all_pass = False
    print(f'  {field} (quarantine) {"":>{18-len(field)}} {status}')

print(f'\nOverall: {"ALL PASS" if all_pass else "VIOLATIONS FOUND"}')

---
## 7. Session Token Analysis

In [ ]:
tokens = q("""
SELECT session_token,
       epoch_id,
       COUNT(*)          AS events,
       MIN(collected_at)::text AS first_seen,
       MAX(collected_at)::text AS last_seen,
       CASE
           WHEN session_token LIKE 'legacy_%%' THEN 'legacy (pre-M7)'
           WHEN LENGTH(session_token) = 64     THEN 'hex-64 (Second Helios)'
           WHEN LENGTH(session_token) = 36     THEN 'uuid (M7)'
           ELSE 'unknown'
       END AS token_type
FROM sp_events
GROUP BY session_token, epoch_id
ORDER BY last_seen DESC
""")

if len(tokens) > 0:
    print(f'{len(tokens)} distinct session_token+epoch pairs')
    print(f'\nToken type breakdown:')
    display(tokens.groupby('token_type')['events'].agg(['count', 'sum']).rename(
        columns={'count': 'sessions', 'sum': 'total_events'}
    ))
    print()
    display(tokens)
else:
    print('No session tokens recorded yet.')

---
## 8. Signal Volume & Domain Coverage

In [ ]:
import matplotlib.pyplot as plt

# Events per day
daily = q("""
SELECT DATE(collected_at) AS day, COUNT(*) AS events
FROM sp_events
GROUP BY DATE(collected_at)
ORDER BY day
""")

if len(daily) > 0:
    daily['day'] = pd.to_datetime(daily['day'])
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(daily['day'], daily['events'], color='steelblue', width=0.8)
    ax.set_title('SpiritPool signals per day')
    ax.set_ylabel('Events')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No daily data yet.')

In [ ]:
# Domain coverage — which job boards are sending signals?
domains = q("""
SELECT payload->>'legacy_domain' AS domain,
       COUNT(*) AS signals,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM sp_events), 1) AS pct
FROM sp_events
GROUP BY payload->>'legacy_domain'
ORDER BY signals DESC
""")

if len(domains) > 0:
    print('Domain coverage:')
    display(domains)
    
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(domains['domain'].fillna('(none)'), domains['signals'], color='steelblue')
    ax.set_xlabel('Signal count')
    ax.set_title('Signals by source domain')
    plt.tight_layout()
    plt.show()
else:
    print('No domain data yet.')

In [ ]:
# Event type breakdown
q("""
SELECT event_type, COUNT(*) AS events
FROM sp_events
GROUP BY event_type
ORDER BY events DESC
""")

---
## 9. Raw Payload Inspector

Drill into individual signals. Change the `LIMIT`/`WHERE` to focus on specific records.

In [ ]:
# Pick a recent signal and show full payload
latest = q("""
SELECT event_id, session_token, collected_at, payload
FROM sp_events
ORDER BY collected_at DESC
LIMIT 3
""")

for _, row in latest.iterrows():
    payload = row['payload']
    if isinstance(payload, str):
        payload = json.loads(payload)
    print(f"=== {row['event_id'][:12]}... | {row['collected_at']} ===")
    print(f"session_token: {row['session_token']}")
    print(json.dumps(payload, indent=2, default=str))
    print()